In [40]:
import pandas as pd

df = pd.read_csv('/home/user1/project/fifa_prediction/df_match_features.csv', parse_dates=['date'])
conf = pd.read_csv('/home/user1/project/fifa_prediction/FIFA_confederations.csv')
df_groups = pd.read_csv('/home/user1/project/fifa_prediction/group_stages.csv', sep=';')
#rank = pd.read_csv('/home/user1/project/fifa_prediction/wc_2026_48_teams_fifa_rank_change_corrected.csv')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [41]:
print(df.shape)
df.head()
#df.tail()


(49122, 12)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,NaN,NaN,NaN
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.801300,1497.198700,5.602600
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1486.622531,1513.377469,-26.754938
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1505.454946,1494.545054,10.909891
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1497.633108,1502.366892,-4.733783


In [42]:
df['home_elo_pre'] = df['home_elo_pre'].fillna(1500)
df['away_elo_pre'] = df['away_elo_pre'].fillna(1500)
df['elo_diff'] = df['home_elo_pre'] - df['away_elo_pre']

print(df.shape)
print("Remaining NaN in Elo:", df[['home_elo_pre', 'away_elo_pre']].isna().sum().sum())

(49122, 12)
Remaining NaN in Elo: 0


In [43]:
df.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.801300,1497.198700,5.602600
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1486.622531,1513.377469,-26.754938
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1505.454946,1494.545054,10.909891
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1497.633108,1502.366892,-4.733783


In [44]:
TIER_1_WORLD_CUP = {'FIFA World Cup'}

TIER_2_CONTINENTAL = {
    'UEFA Euro', 'Copa América', 'African Cup of Nations', 'AFC Asian Cup',
    'Gold Cup', 'CONCACAF Championship', 'Oceania Nations Cup',
    'Confederations Cup',
}

TIER_3_QUALIFIERS_NATIONS = {
    'FIFA World Cup qualification', 'UEFA Euro qualification',
    'African Cup of Nations qualification', 'AFC Asian Cup qualification',
    'Gold Cup qualification', 'CONCACAF Championship qualification',
    'Copa América qualification', 'Oceania Nations Cup qualification',
    'UEFA Nations League', 'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
}

TIER_4_REGIONAL = {
    # Africa
    'CECAFA Cup', 'COSAFA Cup', 'COSAFA Cup qualification', 'WAFF Championship',
    'Amílcar Cabral Cup', 'All-African Games', 'UDEAC Cup', 'UNIFFAC Cup',
    'West African Cup', 'Nile Basin Tournament', 'African Friendship Games',
    # Asia / Oceania
    'Gulf Cup', 'Arab Cup', 'Arab Cup qualification', 'SAFF Cup',
    'AFF Championship', 'AFF Championship qualification', 'EAFF Championship',
    'EAFF Championship qualification', 'ASEAN Championship',
    'ASEAN Championship qualification', 'AFC Challenge Cup',
    'AFC Challenge Cup qualification', 'Asian Games', 'CAFA Nations Cup',
    'Southeast Asian Games', 'South Asian Games', 'Dynasty Cup',
    'Pacific Games', 'South Pacific Games', 'Melanesia Cup',
    'Indian Ocean Island Games', 'Afro-Asian Games',
    # Europe
    'British Home Championship', 'Nordic Championship', 'Baltic Cup',
    'Balkan Cup', 'Central European International Cup',
    # Americas
    'CFU Caribbean Cup', 'CFU Caribbean Cup qualification', 'UNCAF Cup',
    'Central American and Caribbean Games', 'Pan American Championship',
    'CCCF Championship', 'Bolivarian Games', 'NAFC Championship',
    # Multi-sport
    'Olympic Games',
}


In [45]:
def tournament_weight(t):
    if t in TIER_1_WORLD_CUP:          return 5
    if t in TIER_2_CONTINENTAL:        return 4
    if t in TIER_3_QUALIFIERS_NATIONS: return 3
    if t in TIER_4_REGIONAL:           return 2
    return 1   # friendlies + minor exhibitions

df['tournament_weight'] = df['tournament'].apply(tournament_weight)
print(df['tournament_weight'].value_counts().sort_index())


tournament_weight
1    21911
2     6679
3    16177
4     3391
5      964
Name: count, dtype: int64


In [46]:
df['is_competitive'] = df['tournament'] != 'Friendly'
print(df['is_competitive'].value_counts())

is_competitive
True     30837
False    18285
Name: count, dtype: int64


In [47]:
df = df.merge(
    conf.rename(columns={'nation': 'home_team', 'confederation': 'home_confederation'}),
    on='home_team', how='left'
)
df = df.merge(
    conf.rename(columns={'nation': 'away_team', 'confederation': 'away_confederation'}),
    on='away_team', how='left'
)

print("Missing home confederation:", df['home_confederation'].isna().sum())
print("Missing away confederation:", df['away_confederation'].isna().sum())

Missing home confederation: 31774
Missing away confederation: 33082


In [48]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000,1,False,UEFA,UEFA
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.801300,1497.198700,5.602600,1,False,UEFA,UEFA
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1486.622531,1513.377469,-26.754938,1,False,UEFA,UEFA
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1505.454946,1494.545054,10.909891,1,False,UEFA,UEFA
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1497.633108,1502.366892,-4.733783,1,False,UEFA,UEFA


In [49]:
df['home_confederation'] = df['home_confederation'].fillna('Unknown')
df['away_confederation'] = df['away_confederation'].fillna('Unknown')



In [50]:
print("Missing home confederation:", df['home_confederation'].isna().sum())
print("Missing away confederation:", df['away_confederation'].isna().sum())

Missing home confederation: 0
Missing away confederation: 0


In [51]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000,1,False,UEFA,UEFA
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.801300,1497.198700,5.602600,1,False,UEFA,UEFA
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1486.622531,1513.377469,-26.754938,1,False,UEFA,UEFA
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1505.454946,1494.545054,10.909891,1,False,UEFA,UEFA
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1497.633108,1502.366892,-4.733783,1,False,UEFA,UEFA


In [52]:
df.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
49117,2026-06-01,Norway,Sweden,3.0,1.0,Friendly,Oslo,Norway,False,1961.224460,1771.742825,189.481635,1,False,UEFA,UEFA
49118,2026-06-01,Turkey,North Macedonia,4.0,0.0,Friendly,Istanbul,Turkey,False,1954.227181,1646.591037,307.636144,1,False,UEFA,Unknown
49119,2026-06-01,Slovakia,Malta,2.0,1.0,Friendly,Dunajská Streda,Slovakia,False,1725.366386,1326.515389,398.850996,1,False,Unknown,Unknown
49120,2026-06-01,Maldives,Afghanistan,0.0,1.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1225.594413,-189.306181,1,True,Unknown,Unknown
49121,2026-06-01,Maldives,Pakistan,0.0,3.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1030.559852,5.728380,1,True,Unknown,Unknown


In [53]:
cutoff = '2026-06-01'

# Stack home + away into one view (still needed — Brazil is in two columns)
home_view = df[['date', 'home_team', 'home_score', 'away_score', 'tournament']].rename(
    columns={'home_team': 'team', 'home_score': 'gf', 'away_score': 'ga'})
away_view = df[['date', 'away_team', 'away_score', 'home_score', 'tournament']].rename(
    columns={'away_team': 'team', 'away_score': 'gf', 'home_score': 'ga'})
long = pd.concat([home_view, away_view], ignore_index=True)

# Filter: before WC + competitive only
long = long[(long['date'] < cutoff) & (long['tournament'] != 'Friendly')]
long['gd'] = long['gf'] - long['ga']

# For each team, take their last 30 matches and sum
form = (
    long.sort_values('date')
        .groupby('team')
        .tail(30)
        .groupby('team')['gd']
        .sum()
        .reset_index()
        .rename(columns={'gd': 'form_score'})
)

# Keep just the 48 WC teams
df_groups = pd.read_csv('/home/user1/project/fifa_prediction/group_stages.csv', sep=';')
wc_teams = df_groups['nation'].unique()
form = form[form['team'].isin(wc_teams)].sort_values('form_score', ascending=False)

print(form)


                       team  form_score
165                 Morocco        63.0
125                   Japan        61.0
240                   Spain        59.0
198                Portugal        56.0
73                  England        54.0
172             New Zealand        52.0
84                   France        46.0
14                Australia        45.0
225                 Senegal        43.0
108                   Haiti        42.0
3                   Algeria        41.0
170             Netherlands        40.0
183                  Norway        40.0
123             Ivory Coast        37.0
11                Argentina        37.0
270           United States        37.0
23                  Belgium        36.0
116                    Iran        35.0
92                  Germany        35.0
273              Uzbekistan        34.0
127                  Jordan        33.0
237             South Korea        32.0
56                  Croatia        32.0
15                  Austria        32.0


In [54]:
cutoff = '2026-06-01'

# Filter to competitive matches before the World Cup
h2h_data = df[
    (df['date'] < cutoff) &
    (df['tournament'] != 'Friendly')
].copy()

In [55]:
h2h_data.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
49085,2026-03-31,DR Congo,Jamaica,1.0,0.0,FIFA World Cup qualification,Zapopan,Mexico,True,1750.376999,1691.786379,58.590620,3,True,CAF,Unknown
49090,2026-05-26,Nigeria,Zimbabwe,2.0,0.0,Unity Cup,London,England,True,1866.388723,1458.128004,408.260719,1,True,Unknown,Unknown
49091,2026-05-27,India,Jamaica,0.0,2.0,Unity Cup,London,England,True,1250.225947,1675.127514,-424.901567,1,True,Unknown,Unknown
49101,2026-05-30,Zimbabwe,India,1.0,0.0,Unity Cup,London,England,True,1456.169271,1248.431872,207.737399,1,True,Unknown,Unknown
49102,2026-05-30,Nigeria,Jamaica,3.0,0.0,Unity Cup,London,England,True,1868.347457,1676.921589,191.425868,1,True,Unknown,Unknown


In [56]:
h2h_data['gd'] = h2h_data['home_score'] - h2h_data['away_score']

# Direction-agnostic pair key: alphabetical so Argentina vs Brazil == Brazil vs Argentina
h2h_data['team_a'] = h2h_data[['home_team', 'away_team']].min(axis=1)
h2h_data['team_b'] = h2h_data[['home_team', 'away_team']].max(axis=1)

# Convert goal diff to "from team_a's perspective"
# If team_a was home, keep the sign; if team_a was away, flip it
h2h_data['gd_for_a'] = h2h_data.apply(
    lambda r: r['gd'] if r['home_team'] == r['team_a'] else -r['gd'],
    axis=1
)

In [57]:
h2h = (
    h2h_data.sort_values('date')
            .groupby(['team_a', 'team_b'])
            .tail(5)
            .groupby(['team_a', 'team_b'])
            .agg(h2h_score=('gd_for_a', 'sum'),
                 n_meetings_analysed=('gd_for_a', 'size'))
            .reset_index()
)


In [58]:
h2h.loc[h2h['n_meetings_analysed'] < 3, 'h2h_score'] = 0


In [59]:
wc_teams = set(df_groups['nation'])
h2h = h2h[h2h['team_a'].isin(wc_teams) & h2h['team_b'].isin(wc_teams)]

print(f"Pairings with history: {len(h2h)}")
print("\nTop 10 most lopsided rivalries:")
print(h2h.reindex(h2h['h2h_score'].abs().sort_values(ascending=False).index).head(10))


Pairings with history: 565

Top 10 most lopsided rivalries:
                      team_a         team_b  h2h_score  n_meetings_analysed
400                Argentina  United States       18.0                    5
594                  Austria          Spain      -17.0                    5
4488                  Mexico   Saudi Arabia       14.0                    5
2012                 Curaçao         Mexico      -14.0                    5
880                  Belgium       Scotland       13.0                    5
2957                 Germany         Norway       13.0                    4
1094  Bosnia and Herzegovina       Portugal      -13.0                    5
5007                Portugal    Switzerland       12.0                    5
5324                   Spain         Turkey       11.0                    5
5008                Portugal         Turkey       10.0                    5


In [60]:
print(h2h[(h2h['team_a'] == 'Argentina') & (h2h['team_b'] == 'Brazil')])
print("------------------------------------------------------------------")
print(h2h[(h2h['team_a'] == 'Algeria') & (h2h['team_b'] == 'Morocco')])
print("------------------------------------------------------------------")
print(h2h[(h2h['team_a'] == 'France') & (h2h['team_b'] == 'Senegal')])
print("------------------------------------------------------------------")
print(h2h[(h2h['team_a'] == 'France') & (h2h['team_b'] == 'Spain')])


        team_a  team_b  h2h_score  n_meetings_analysed
353  Argentina  Brazil        6.0                    5
------------------------------------------------------------------
      team_a   team_b  h2h_score  n_meetings_analysed
147  Algeria  Morocco       -6.0                    5
------------------------------------------------------------------
      team_a   team_b  h2h_score  n_meetings_analysed
2782  France  Senegal        0.0                    2
------------------------------------------------------------------
      team_a team_b  h2h_score  n_meetings_analysed
2788  France  Spain       -2.0                    5


In [61]:
df[
    (((df['home_team'] == 'France') & (df['away_team'] == 'Senegal')) |
     ((df['home_team'] == 'Senegal') & (df['away_team'] == 'France'))) &
    (df['tournament'] != 'Friendly')
][['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']]


,date,home_team,away_team,home_score,away_score,tournament
5817,1963-04-18,Senegal,France,2.0,0.0,African Friendship Games
26436,2002-05-31,France,Senegal,0.0,1.0,FIFA World Cup


In [62]:
print("df shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nNaN counts:")
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 0] if nan_counts.sum() > 0 else "✓ No NaN values")


df shape: (49122, 16)

Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'home_elo_pre', 'away_elo_pre', 'elo_diff', 'tournament_weight', 'is_competitive', 'home_confederation', 'away_confederation']

NaN counts:
✓ No NaN values


In [63]:
df.tail()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
49117,2026-06-01,Norway,Sweden,3.0,1.0,Friendly,Oslo,Norway,False,1961.224460,1771.742825,189.481635,1,False,UEFA,UEFA
49118,2026-06-01,Turkey,North Macedonia,4.0,0.0,Friendly,Istanbul,Turkey,False,1954.227181,1646.591037,307.636144,1,False,UEFA,Unknown
49119,2026-06-01,Slovakia,Malta,2.0,1.0,Friendly,Dunajská Streda,Slovakia,False,1725.366386,1326.515389,398.850996,1,False,Unknown,Unknown
49120,2026-06-01,Maldives,Afghanistan,0.0,1.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1225.594413,-189.306181,1,True,Unknown,Unknown
49121,2026-06-01,Maldives,Pakistan,0.0,3.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1030.559852,5.728380,1,True,Unknown,Unknown


In [64]:
print(f"\nform table: {form.shape[0]} teams (expected 48)")
assert form.shape[0] == 48, "Missing teams in form table"



form table: 48 teams (expected 48)


In [65]:
print(f"h2h table: {h2h.shape[0]} pairings between WC2026 teams")

h2h table: 565 pairings between WC2026 teams


In [66]:
df.to_csv('/home/user1/project/fifa_prediction/processed/df_match_features.csv', index=False)
form.to_csv('/home/user1/project/fifa_prediction/processed/df_form_2026.csv', index=False)
h2h.to_csv('/home/user1/project/fifa_prediction/processed/df_h2h_2026.csv', index=False)

print("\n✓ Saved all three files to /processed/")


✓ Saved all three files to /processed/
